In [404]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error

from xgboost import XGBRegressor

In [405]:
df = pd.read_csv("../data/raw/2022/DataPublication_final/GroundTruth/HYBRID_HIPS_V3.5_ALLPLOTS.csv")
df.head()

,index,qrCode,location,irrigationProvided,nitrogenTreatment,poundsOfNitrogenPerAcre,experiment,plotLength,block,row,range,plotNumber,genotype,plantingDate,totalStandCount,daysToAnthesis,GDDToAnthesis,yieldPerAcre
0,8,NaN,Ames,NaN,NaN,0,4231,NaN,NaN,2,23,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,11,NaN,Ames,NaN,NaN,0,4231,NaN,NaN,2,31,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0,NaN,Ames,NaN,NaN,0,4231,NaN,NaN,3,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,9,NaN,Ames,NaN,NaN,0,4231,NaN,NaN,3,29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,12,NaN,Ames,NaN,NaN,0,4231,NaN,NaN,3,31,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [406]:
print("Shape before cleaning:", df.shape)
print("Missing yield values:", df["yieldPerAcre"].isna().sum())

Shape before cleaning: (2291, 18)
Missing yield values: 160


In [407]:
df = df.dropna(subset=["yieldPerAcre"]).copy()

print("Shape after dropping missing target:", df.shape)

Shape after dropping missing target: (2131, 18)


In [408]:
columns_to_drop_if_present = [
    "plotNumber",
    "qrCode",
    "totalStandCount",
    "daysToAnthesis",
    "GDDToAnthesis"
]

existing_drop_cols = [col for col in columns_to_drop_if_present if col in df.columns]

df = df.drop(columns=existing_drop_cols)

print("Dropped columns:", existing_drop_cols)
print("Shape after column drop:", df.shape)

Dropped columns: ['plotNumber', 'qrCode', 'totalStandCount', 'daysToAnthesis', 'GDDToAnthesis']
Shape after column drop: (2131, 13)


In [409]:
target_col = "yieldPerAcre"

X = df.drop(columns=[target_col])
y = df[target_col]


X = engineer_features(X)

print("X shape after engineering:", X.shape)
print("X columns after engineering:\n", X.columns.tolist())

X shape after engineering: (2131, 10)
X columns after engineering:
 ['irrigationProvided', 'nitrogenTreatment', 'poundsOfNitrogenPerAcre', 'plotLength', 'genotype', 'planting_month', 'planting_dayofyear', 'is_early_season', 'nitrogen_x_irrigation', 'nitrogen_squared']


In [410]:
def engineer_features(X: pd.DataFrame, top_locations=None) -> pd.DataFrame:
    X = X.copy()

    if "plantingDate" in X.columns:
        X["plantingDate"] = pd.to_datetime(X["plantingDate"], errors="coerce")
        X["planting_month"] = X["plantingDate"].dt.month
        X["planting_dayofyear"] = X["plantingDate"].dt.dayofyear
        X["is_early_season"] = (X["planting_dayofyear"] < 150).astype(int)
        X = X.drop(columns=["plantingDate"])

    if "experiment" in X.columns:
        X = X.drop(columns=["experiment"])

    if "poundsOfNitrogenPerAcre" in X.columns and "irrigationProvided" in X.columns:
        X["nitrogen_x_irrigation"] = (
            X["poundsOfNitrogenPerAcre"] * X["irrigationProvided"]
        )

    if "poundsOfNitrogenPerAcre" in X.columns:
        X["nitrogen_squared"] = X["poundsOfNitrogenPerAcre"] ** 2

    if "location" in X.columns:
        X= X.drop(columns=["location"])
    
    drop_cols = ["index", "row", "range", "block", "plotLength"]

    for col in drop_cols:
        if col in X.columns:
            X = X.drop(columns=[col])

    return X

In [411]:
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

high_cardinality = [
    col for col in categorical_features
    if X[col].nunique() > 50
]

print("High-cardinality columns:", high_cardinality)

X = X.drop(columns=high_cardinality)

print("New shape after dropping:", X.shape)

High-cardinality columns: ['genotype']
New shape after dropping: (2131, 9)


In [412]:
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Number of numeric columns:", len(numeric_features))
print("Number of categorical columns:", len(categorical_features))
print("\nSample numeric columns:", numeric_features[:10])
print("\nSample categorical columns:", categorical_features[:10])

Number of numeric columns: 8
Number of categorical columns: 1

Sample numeric columns: ['irrigationProvided', 'poundsOfNitrogenPerAcre', 'plotLength', 'planting_month', 'planting_dayofyear', 'is_early_season', 'nitrogen_x_irrigation', 'nitrogen_squared']

Sample categorical columns: ['nitrogenTreatment']


In [413]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

In [414]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", max_categories=10))
])

In [415]:
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [416]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (1704, 9)
X_test shape: (427, 9)


In [417]:
model = XGBRegressor(
    n_estimators=1200,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42
)

In [418]:
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

In [419]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
import numpy as np

gkf = GroupKFold(n_splits=5)
groups = df.loc[X.index, "location"]

scores = []

for train_idx, test_idx in gkf.split(X, y, groups=groups):
    pipeline.fit(X.iloc[train_idx], y.iloc[train_idx])
    preds = pipeline.predict(X.iloc[test_idx])
    rmse = np.sqrt(mean_squared_error(y.iloc[test_idx], preds))
    scores.append(rmse)

print("Group CV RMSE scores:", scores)
print("Mean Group CV RMSE:", np.mean(scores))
print("Std Group CV RMSE:", np.std(scores))

Group CV RMSE scores: [91.97966182563006, 39.81332558465486, 51.654649170092355, 95.19013693445339, 32.64319330315783]
Mean Group CV RMSE: 62.256193363597696
Std Group CV RMSE: 26.310203505679343


In [420]:
pipeline.fit(X_train, y_train)
print("Training complete.")

Training complete.


In [421]:
preds = pipeline.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, preds))

print("RMSE:", rmse)

RMSE: 28.213230235691473


In [422]:
results = pd.DataFrame({
    "actual": y_test.values,
    "predicted": preds
})

results.head(10)

,actual,predicted
0,152.83,158.999817
1,183.96,121.619118
2,8.59,27.761925
3,12.65,27.761925
4,82.19,121.619118
5,159.98,158.999817
6,36.91,27.761925
7,81.10,88.264061
8,151.44,145.502533
9,153.77,158.999817


In [423]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    pipeline,
    X,
    y,
    scoring="neg_mean_squared_error",
    cv=5
)

rmse_scores = np.sqrt(-scores)

print("CV RMSE scores:", rmse_scores)
print("Mean CV RMSE:", rmse_scores.mean())

CV RMSE scores: [69.61436793 53.56347436 49.63319051 29.81551378 69.82694091]
Mean CV RMSE: 54.49069749895746


In [424]:
feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
importances = pipeline.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

importance_df.head(20)

,feature,importance
4,num__planting_dayofyear,0.338903
10,cat__nitrogenTreatment_Medium,0.194050
9,cat__nitrogenTreatment_Low,0.170210
6,num__nitrogen_x_irrigation,0.061471
7,num__nitrogen_squared,0.059265
3,num__planting_month,0.057754
2,num__plotLength,0.040585
1,num__poundsOfNitrogenPerAcre,0.040010
0,num__irrigationProvided,0.024601
8,cat__nitrogenTreatment_High,0.013151
